# 04 · Execution, runtime guardrails and step records

**Role of this layer.** The executor is the only code that calls capability
functions. It re-validates the plan (it does not trust its caller), refuses to
start while any BLOCK exists or any APPROVAL issue is unapproved, then runs the
steps in order with the runtime guardrails that need data:

| code | severity | checks |
|---|---|---|
| R01 | BLOCK | snapshot file hash still matches the catalog |
| R02 | BLOCK | enough observations for the capability (e.g. 3 x window, 20 per group) |
| R03 | WARN | more than 5% of a raw series missing |
| R04 | WARN | an inner join drops more than 10% of an input |
| R05 | WARN | fewer than 5 distinct flagged episodes |

Every step produces a **step record**: parameters used (defaults filled), input
hashes, output hash, sample size and range, duration, issues.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import load_catalog
from quantlab.executor import execute_plan
from quantlab.plan import Plan
from quantlab.scenarios import SCENARIOS_BY_ID
from quantlab.validation import fill_defaults

catalog = load_catalog(settings.catalog_path)


def scenario(scenario_id):
    s = SCENARIOS_BY_ID[scenario_id]
    return fill_defaults(Plan(plan_id="demo", request=s.request, **s.plan))


plan_a = scenario("A")
refused = execute_plan(plan_a, catalog, approved_issue_ids=set())
print(refused.status, "-", refused.error)

blocked - Approval required for: G12:s3:return_method


With the approval granted, scenario A runs on the real NASDAQ snapshot:

In [3]:
result = execute_plan(plan_a, catalog, approved_issue_ids={"G12:s3:return_method"})
print("status:", result.status, f"({result.duration_ms:.0f} ms)")
pd.DataFrame(
    [
        {
            "step": r.step_id,
            "capability": r.capability,
            "params": r.params,
            "n_obs": r.n_obs,
            "range": f"{r.start}..{r.end}",
            "ms": r.duration_ms,
            "output sha256": r.output_hash[:12],
        }
        for r in result.step_records
    ]
)

status: completed (275 ms)


,step,capability,params,n_obs,range,ms,output sha256
0,s1,load_series,{'dataset_id': 'NASDAQCOM'},14040,1971-02-05..2026-10-13,114.73,3ea1bb2ab2fe
1,s2,resample,"{'freq': 'W', 'how': 'last'}",2906,1971-02-05..2026-10-09,53.78,0c44f38c8ce1
2,s3,compute_returns,"{'method': 'simple', 'periods': 1}",2905,1971-02-12..2026-10-09,22.10,b607a740b787
3,s4,rolling_volatility,"{'window': 12, 'annualize': True}",2894,1971-04-30..2026-10-09,22.08,35c7d923a748
4,s5,flag_regime,"{'quantile': 0.9, 'method': 'expanding', 'min_periods': 52}",2842,1972-04-28..2026-10-09,17.70,710fc3814adb
5,s6,forward_return,{'horizon': 4},2901,1971-02-12..2026-09-11,27.39,a42273a39d05
6,s7,compare_groups,"{'test': 'welch_t', 'nonoverlapping': True}",710,1972-04-28..2026-09-04,7.10,83833d2d7cf3
7,s8,ols_hac,{'maxlags': None},2838,1972-04-28..2026-09-11,10.20,a4f48912e3ed


The two tests at the end of the plan. The Welch t-test uses non-overlapping
4-week returns; the Newey-West regression keeps every overlapping observation
and corrects the standard errors with horizon - 1 = 3 lags instead.

In [4]:
for step_id in ["s7", "s8"]:
    value = result.artifacts[step_id].value
    print(
        step_id,
        value["test"],
        "statistic",
        round(value["statistic"], 4),
        "p-value",
        round(value["p_value"], 4),
    )
    print("   ", value["details"])

s7 welch_t statistic 0.675 p-value 0.5009
    {'n_flagged': 112, 'n_normal': 598, 'mean_flagged': 0.01445785189021451, 'mean_normal': 0.008389224369524209, 'diff_means': 0.006068627520690302, 'n_flagged_episodes': 30, 'nonoverlapping': True, 'horizon': 4}
s8 ols_hac statistic 0.6798 p-value 0.4966
    {'coef_flag': 0.004745678750434298, 'se_flag': 0.006981157172241879, 't_flag': 0.6797839718182859, 'const': 0.008504563695949433, 'n': 2838, 'n_flagged': 458, 'n_flagged_episodes': 42, 'maxlags': 3, 'maxlags_source': 'horizon - 1', 'horizon': 4}


**R02**: a 200-month window on monthly data does not have 3 x 200 observations.

In [5]:
plan = scenario("A")
plan.steps[1].params["freq"] = "M"
plan.steps[3].params["window"] = 200
blocked = execute_plan(plan, catalog, {"G12:s3:return_method"})
print(blocked.status, "-", blocked.error)

completed - None


**R04** fires on the real data in scenario F: the 2-year yield starts in 1976 and
the 10-year in 1962, so the spread silently loses 22% of the 10-year history.
The run completes, with the warning recorded.

In [6]:
result_f = execute_plan(scenario("F"), catalog)
print(result_f.status)
for issue in result_f.issues:
    print(f"[{issue.severity}] {issue.code}: {issue.message}")

completed
[WARN] R04: Aligning the inputs kept 2628 of 3380 rows of 'a' (22% dropped).


Determinism: running the same plan twice gives identical output hashes.

In [7]:
again = execute_plan(scenario("F"), catalog)
[r.output_hash for r in again.step_records] == [r.output_hash for r in result_f.step_records]

True